In [ ]:
import logging

import openeo.processes
from utils import urls

import openeo

logging.basicConfig(level=logging.INFO)

In [ ]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")

In [ ]:
connection.authenticate_oidc()

# Parameters

In [ ]:
spatial_extent = {
    "west": 30.55,
    "south": 1.07,
    "east": 31.23,
    "north": 1.55,
}

In [ ]:
# minimum canopy cover to be considered forest
# should be one of (10, 20, 30, 40, 50, 60, 70, 80, 90)
# a value of 30 means > 30% canopy cover
canopy_cover_threshold = 30

# minimum likelihood to be considered natural forest
natural_forest_threshold = 0.08

# minimum connected area to be considered forest (m^2)
min_connected_area = 10000

In [ ]:
spatial_resolution = 30  # m

In [ ]:
temporal_variability_threshold = 0.5
flattening_threshold = 0.12
logistic_sse_threshold = 18.3

In [ ]:
# more restrictive (excludes more pixels from being detected as deforestation) than default
temporal_variability_threshold = 0.6
flattening_threshold = 0.14

In [ ]:
cropland_probability_threshold = 0.1

# Script

In [ ]:
# collect outputs as we go, to built a multi-result process graph
process_graph_results = []

In [ ]:
forest_baseline_datacube = connection.datacube_from_process(
    "forest_baseline",
    namespace=urls.FOREST_BASELINE_UDP,
    spatial_extent=spatial_extent,
    canopy_cover_threshold=canopy_cover_threshold,
    natural_forest_threshold=natural_forest_threshold,
    min_connected_area=min_connected_area,
)

In [ ]:
process_graph_results.append(
    forest_baseline_datacube.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0010_forest_baseline_datacube",
        },
    )
)
process_graph_results.append(
    forest_baseline_datacube.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0010_forest_baseline_datacube",
        },
    )
)

In [ ]:
sentinel_1_datacube = connection.datacube_from_process(
    "s1_logistic_processing",
    namespace=urls.S1_PROCESSING_UDP,
    spatial_extent=spatial_extent,
    spatial_resolution=spatial_resolution,
)

In [ ]:
process_graph_results.append(
    sentinel_1_datacube.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0020_sentinel_1_datacube",
        },
    )
)
process_graph_results.append(
    sentinel_1_datacube.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0020_sentinel_1_datacube",
        },
    )
)

In [ ]:
decimal_year_of_deforestation_datacube = connection.datacube_from_process(
    "deforestation",
    namespace=urls.DEFORESTATION_UDP,
    # spatial_extent=spatial_extent,
    forest_baseline_datacube=forest_baseline_datacube,
    sentinel_1_datacube=sentinel_1_datacube,
    spatial_extent=spatial_extent,
    spatial_resolution=spatial_resolution,
    temporal_variability_threshold=temporal_variability_threshold,
    flattening_threshold=flattening_threshold,
    logistic_sse_threshold=logistic_sse_threshold,
    min_connected_area=min_connected_area,
)

In [ ]:
process_graph_results.append(
    decimal_year_of_deforestation_datacube.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0030_decimal_year_of_deforestation_datacube",
        },
    )
)
process_graph_results.append(
    decimal_year_of_deforestation_datacube.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0030_decimal_year_of_deforestation_datacube",
        },
    )
)

In [ ]:
kpis_vector_cube = connection.datacube_from_process(
    "KPIs",
    namespace=urls.KPIS_UDP,
    decimal_year_of_deforestation_datacube=decimal_year_of_deforestation_datacube,
    spatial_extent=spatial_extent,
    spatial_resolution=spatial_resolution,
    cropland_probability_threshold=cropland_probability_threshold,
)

In [ ]:
process_graph_results.append(
    kpis_vector_cube.save_result(
        format="Parquet",
        options={
            "filename_prefix": "0040_kpis_vector_cube",
        },
    )
)

# Run job

In [ ]:
multi_result = openeo.MultiResult(process_graph_results)
job = multi_result.create_job()

In [ ]:
# https://forum.dataspace.copernicus.eu/t/error-chaining-upds/5528
job = kpis_vector_cube.save_result(
    format="Parquet",
    options={
        "filename_prefix": "0040_kpis_vector_cube",
    },
).create_job()

In [ ]:
job.start_and_wait()
# 1 hr 15 min, cost 143 credits

In [ ]:
results = job.get_results()

In [ ]:
!mkdir -p output-script/
!rm -r output-script/

In [ ]:
results.download_files("output-script/")

In [ ]:
import json

with open("logs.json", "w") as f:
    json.dump(job.logs(), f, indent=2)